# Logistic Regression

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.ensemble import StackingClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')


## Load Data

In [3]:
# 1. LOAD DỮ LIỆU ĐÃ SCALED
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
valid_df = pd.read_csv(os.path.join(data_dir, 'valid_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]


### Danh sách lưu kết quả để xuất CSV

In [4]:
results_list = []

def evaluate_model(model, name, X_split, y_split, split_name):
    """Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)."""
    y_pred = model.predict(X_split)
    y_prob = model.predict_proba(X_split)[:, 1]  # Xác suất class 1 để tính AUC

    acc = accuracy_score(y_split, y_pred)
    prec = precision_score(y_split, y_pred)
    rec = recall_score(y_split, y_pred)
    f1 = f1_score(y_split, y_pred)
    auc = roc_auc_score(y_split, y_prob)

    res = {
        'Algorithm': name,
        'Split': split_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'AUC': round(auc, 4)
    }

    results_list.append(res)
    print(f"[{name} | {split_name}] Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    return res


In [5]:
# K-FOLD CONFIG
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_kfold(model, name, X_data, y_data, cv_split):
    """Đánh giá K-Fold và lưu kết quả trung bình vào results_list."""
    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'auc': 'roc_auc'
    }

    cv_results = cross_validate(
        model,
        X_data,
        y_data,
        cv=cv_split,
        scoring=scoring,
        n_jobs=-1
    )

    res = {
        'Algorithm': name,
        'Split': 'KFold',
        'Accuracy': round(cv_results['test_accuracy'].mean(), 4),
        'Precision': round(cv_results['test_precision'].mean(), 4),
        'Recall': round(cv_results['test_recall'].mean(), 4),
        'F1-Score': round(cv_results['test_f1'].mean(), 4),
        'AUC': round(cv_results['test_auc'].mean(), 4)
    }

    results_list.append(res)
    print(
        f"[{name} | KFold] Acc: {res['Accuracy']:.4f} | Prec: {res['Precision']:.4f} | "
        f"Rec: {res['Recall']:.4f} | F1: {res['F1-Score']:.4f} | AUC: {res['AUC']:.4f}"
    )
    return res


### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [6]:
print("--- V1: Baseline Logistic Regression ---")
LogisticRegressionModel_baseline_model = LogisticRegression(random_state=42, max_iter=1000)

# Đánh giá K-Fold trước
evaluate_kfold(LogisticRegressionModel_baseline_model, "V1: Logistic Regression Baseline", X_train, y_train, kfold)

# Train cố định trên train set rồi đánh giá validation/test
LogisticRegressionModel_baseline_model.fit(X_train, y_train)
evaluate_model(LogisticRegressionModel_baseline_model, "V1: Logistic Regression Baseline", X_valid, y_valid, "Validation")
evaluate_model(LogisticRegressionModel_baseline_model, "V1: Logistic Regression Baseline", X_test, y_test, "Test")


--- V1: Baseline Logistic Regression ---
[V1: Logistic Regression Baseline | KFold] Acc: 0.9919 | Prec: 0.9857 | Rec: 0.9971 | F1: 0.9914 | AUC: 0.9998
[V1: Logistic Regression Baseline | Validation] Acc: 0.9951 | Prec: 0.9892 | Rec: 1.0000 | F1: 0.9946 | AUC: 1.0000
[V1: Logistic Regression Baseline | Test] Acc: 0.9877 | Prec: 0.9734 | Rec: 1.0000 | F1: 0.9865 | AUC: 0.9999


{'Algorithm': 'V1: Logistic Regression Baseline',
 'Split': 'Test',
 'Accuracy': 0.9877,
 'Precision': 0.9734,
 'Recall': 1.0,
 'F1-Score': 0.9865,
 'AUC': 0.9999}

### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [7]:
print("--- V2: GridSearchCV Tuning ---")
param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'solver': ['lbfgs', 'liblinear'], 'penalty': ['l2']}

grid_search = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
LogisticRegressionModel_tuned_model = grid_search.best_estimator_

print(f"Best Params found by CV: {grid_search.best_params_}")

# Đánh giá K-Fold cho mô hình tốt nhất
evaluate_kfold(LogisticRegressionModel_tuned_model, "V2: Logistic Regression Tuned (GridSearch)", X_train, y_train, kfold)

evaluate_model(LogisticRegressionModel_tuned_model, "V2: Logistic Regression Tuned (GridSearch)", X_valid, y_valid, "Validation")
evaluate_model(LogisticRegressionModel_tuned_model, "V2: Logistic Regression Tuned (GridSearch)", X_test, y_test, "Test")


--- V2: GridSearchCV Tuning ---
Best Params found by CV: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
[V2: Logistic Regression Tuned (GridSearch) | KFold] Acc: 0.9946 | Prec: 0.9912 | Rec: 0.9969 | F1: 0.9940 | AUC: 0.9998
[V2: Logistic Regression Tuned (GridSearch) | Validation] Acc: 0.9951 | Prec: 0.9892 | Rec: 1.0000 | F1: 0.9946 | AUC: 1.0000
[V2: Logistic Regression Tuned (GridSearch) | Test] Acc: 0.9877 | Prec: 0.9734 | Rec: 1.0000 | F1: 0.9865 | AUC: 1.0000


{'Algorithm': 'V2: Logistic Regression Tuned (GridSearch)',
 'Split': 'Test',
 'Accuracy': 0.9877,
 'Precision': 0.9734,
 'Recall': 1.0,
 'F1-Score': 0.9865,
 'AUC': 1.0}

In [8]:
print("--- V3: Ensemble Learning (Stacking) ---")
# Kết hợp mô hình chính và KNN làm Base Models
base_estimators = [('lr', LogisticRegression(random_state=42, max_iter=1000)), ('knn', KNeighborsClassifier(n_neighbors=5))]

LogisticRegressionModel_stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=kfold
)

# Đánh giá K-Fold cho stacking
evaluate_kfold(LogisticRegressionModel_stacking_model, "V3: Logistic Regression Stacking Ensemble", X_train, y_train, kfold)

LogisticRegressionModel_stacking_model.fit(X_train, y_train)
evaluate_model(LogisticRegressionModel_stacking_model, "V3: Logistic Regression Stacking Ensemble", X_valid, y_valid, "Validation")
evaluate_model(LogisticRegressionModel_stacking_model, "V3: Logistic Regression Stacking Ensemble", X_test, y_test, "Test")


--- V3: Ensemble Learning (Stacking) ---
[V3: Logistic Regression Stacking Ensemble | KFold] Acc: 0.9959 | Prec: 0.9912 | Rec: 1.0000 | F1: 0.9955 | AUC: 0.9999
[V3: Logistic Regression Stacking Ensemble | Validation] Acc: 0.9951 | Prec: 0.9892 | Rec: 1.0000 | F1: 0.9946 | AUC: 1.0000
[V3: Logistic Regression Stacking Ensemble | Test] Acc: 0.9926 | Prec: 0.9839 | Rec: 1.0000 | F1: 0.9919 | AUC: 1.0000


{'Algorithm': 'V3: Logistic Regression Stacking Ensemble',
 'Split': 'Test',
 'Accuracy': 0.9926,
 'Precision': 0.9839,
 'Recall': 1.0,
 'F1-Score': 0.9919,
 'AUC': 1.0}

### (5) Lưu kết quả

In [9]:
# CẤU HÌNH ĐƯỜNG DẪN LƯU
save_path = '../experiment/Logistic_Regression/'
os.makedirs(save_path, exist_ok=True)

# 1. TỔNG HỢP & LƯU 1 FILE CSV DUY NHẤT
df_results = pd.DataFrame(results_list)

csv_output = os.path.join(save_path, 'Logistic_Regression_full_results.csv')
df_results.to_csv(csv_output, index=False)

# 2. LƯU MODELS (.pkl)
with open(os.path.join(save_path, 'Logistic_Regression_baseline.pkl'), 'wb') as f:
    pickle.dump(LogisticRegressionModel_baseline_model, f)

with open(os.path.join(save_path, 'Logistic_Regression_tuned.pkl'), 'wb') as f:
    pickle.dump(LogisticRegressionModel_tuned_model, f)

with open(os.path.join(save_path, 'Logistic_Regression_stacking.pkl'), 'wb') as f:
    pickle.dump(LogisticRegressionModel_stacking_model, f)

print("-" * 40)
print(f"✅ Đã lưu tất cả kết quả vào: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)


----------------------------------------
✅ Đã lưu tất cả kết quả vào: ../experiment/Logistic_Regression/Logistic_Regression_full_results.csv
✅ Đã lưu models tại: ../experiment/Logistic_Regression/
----------------------------------------


,Algorithm,Split,Accuracy,Precision,Recall,F1-Score,AUC
0,V1: Logistic Regression Baseline,KFold,0.9919,0.9857,0.9971,0.9914,0.9998
1,V1: Logistic Regression Baseline,Validation,0.9951,0.9892,1.0000,0.9946,1.0000
2,V1: Logistic Regression Baseline,Test,0.9877,0.9734,1.0000,0.9865,0.9999
3,V2: Logistic Regression Tuned (GridSearch),KFold,0.9946,0.9912,0.9969,0.9940,0.9998
4,V2: Logistic Regression Tuned (GridSearch),Validation,0.9951,0.9892,1.0000,0.9946,1.0000
5,V2: Logistic Regression Tuned (GridSearch),Test,0.9877,0.9734,1.0000,0.9865,1.0000
6,V3: Logistic Regression Stacking Ensemble,KFold,0.9959,0.9912,1.0000,0.9955,0.9999
7,V3: Logistic Regression Stacking Ensemble,Validation,0.9951,0.9892,1.0000,0.9946,1.0000
8,V3: Logistic Regression Stacking Ensemble,Test,0.9926,0.9839,1.0000,0.9919,1.0000


In [10]:
!jupyter nbconvert --to html Logistic_Regression_model.ipynb

[NbConvertApp] Converting notebook Logistic_Regression_model.ipynb to html
[NbConvertApp] Writing 308562 bytes to Logistic_Regression_model.html
